# StoCFL com rótulos non-IID (Dirichlet): escolha de τ por Monte Carlo

Escolhe o `τ` da grade do [plano do dropout com treino](../../docs/plano-dropout-treino-stocfl.md) (passo 3, seção 6) por **Monte Carlo sobre `τ`**: sorteia `N_TAU` valores de `τ` uniformes em `[TAU_MIN, TAU_MAX]` e, para cada um, roda a clusterização com **todos os outros parâmetros da grade** (`n = 240`, `α` = 0.5 e 1.0, seeds 0 a 9, `m` = 100% e 50%, sem dropout e com `d` = 0.05 e 0.1, `--drop_rounds 10`, 50 rodadas). O `τ` escolhido é o de **maior ARI médio** contra um gabarito, sobre as 120 combinações (D-046 em [decisoes.md](../../docs/decisoes.md)).

- **Gabarito.** O Dirichlet puro não tem grupos reais. O gabarito é construído: um **agrupamento hierárquico (Ward) das distribuições de rótulos** dos clientes, com o nº de grupos escolhido pela maior **silhueta** (`k` de 2 a 40). É o que o `Ψ` deveria recuperar se refletisse só a composição de classes; **não é uma verdade**.
- **Simulador.** Com `--train 0` a partição depende só dos `Ψ` dos clientes e de quem é sorteado em cada rodada. O notebook calcula os `Ψ` com o **script real** (uma execução de 1 rodada com `m = 100%` por `α` e seed, que grava `psi.pkl`) e guarda a matriz de Gram `G = ΨΨᵀ` (o cosseno entre clientes). O cosseno entre os centros de dois clusters sai de `G` (`S_AB / √(S_AA·S_BB)`, com `S_AB` a soma do bloco), e a fusão é a do script: todos os pares de clusters sorteados com cosseno `> τ` se unem (componentes conexas), e o cluster fica com o menor id. O sorteio repete o do script (`random.seed(seed)` e `Availability`). A seção 4 **valida** o simulador contra execuções reais do script de 50 rodadas, rodada a rodada.
- **Política 7.** As execuções gravam logs em `logs/`; o JSON `logs/resultados-varredura-tau-dirichlet.json` guarda o que importa (as matrizes `G`, os histogramas, o gabarito, os sorteios, os resultados do Monte Carlo e da validação). Com `DADOS_JSON = True` o notebook só lê o JSON e plota. Sem os logs e com o JSON destes parâmetros já gravado, ele para com um aviso em vez de rodar tudo de novo (`REEXECUTAR = True` força).
- **Ambiente:** o kernel é o `.venv` da raiz; abra o notebook a partir de `projetos/StoCFL/`. As partições Dirichlet ficam em `datasets/rotated_mnist_240_<seed>_k1_a<α>/` (geradas na primeira execução, 228 MB cada).

**Custo:** 20 execuções de 1 rodada (~1 min e 1,5 GB de `psi.pkl` cada), o Monte Carlo em paralelo na CPU (alguns minutos) e as execuções de validação (50 rodadas, alguns minutos cada).

In [ ]:
import json
import multiprocessing
import os
import random
import re
import shutil
import subprocess
import sys
import time
from concurrent.futures import ProcessPoolExecutor
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import torch
from plotly.subplots import make_subplots
from scipy.cluster.hierarchy import fcluster, linkage
from sklearn.metrics import adjusted_rand_score, silhouette_score
from threadpoolctl import threadpool_limits

PROJETO = Path.cwd()
SCRIPT = "StoCFL-FCC+CFL.py"
assert (PROJETO / SCRIPT).exists(), "abra o notebook a partir de projetos/StoCFL/"
from availability import Availability

RESULTADOS_JSON = PROJETO / "logs" / "resultados-varredura-tau-dirichlet.json"

# Parâmetros da grade (plano, seção 2)
N = 240                           # nº de clientes
ALPHAS = [0.5, 1.0]               # α do Dirichlet (--dirichlet_alpha)
SEEDS = list(range(10))           # a seed define a partição, o modelo inicial (e os Ψ) e o sorteio
ROUNDS = 50                       # rodadas de comunicação
MU = 0.05                         # só entra no nome da pasta de logs (sem treino)
FRACOES = [1.0, 0.5]              # m: fração dos clientes sorteada por rodada
DROPS = [0.0, 0.05, 0.1]          # d (--drop); 0.0 = sem dropout
DROP_ROUNDS = 10                  # L_max (--drop_rounds)

# Monte Carlo sobre τ
N_TAU = 400                       # nº de valores de τ sorteados
TAU_MIN, TAU_MAX = 0.3, 1.0       # τ ~ U[TAU_MIN, TAU_MAX]; abaixo de ~0.5 tudo vira 1 cluster (plano, seção 4.3)
SEMENTE_MC = 0                    # semente do sorteio dos τ
K_GABARITO = list(range(2, 41))   # nº de grupos testados no gabarito (silhueta)
TAU_PSI = 1.0                     # τ das execuções de 1 rodada que só calculam os Ψ (nenhuma fusão)

# Validação do simulador contra o script real (50 rodadas, --train 0): (α, seed, m, d) × τ; o τ escolhido é acrescentado
VALIDACAO = [(0.5, 0, 0.5, 0.1), (0.5, 0, 1.0, 0.05), (1.0, 1, 0.5, 0.1), (1.0, 1, 1.0, 0.05)]
VALIDACAO_TAUS = [0.75, 0.85]
VALIDACAO_ESCOLHIDO = (0.5, 2, 0.5, 0.1)   # execução extra com o τ escolhido

PROCESSOS = min(20, os.cpu_count())
REEXECUTAR = False                # True ignora execuções já concluídas e roda de novo
DADOS_JSON = False                # True não lê logs nem roda nada: carrega tudo de RESULTADOS_JSON e só plota


def nome(d):
    return "sem dropout" if d == 0 else f"d = {d:g}"


def parametros_atuais():
    return {"N": N, "ALPHAS": ALPHAS, "SEEDS": SEEDS, "ROUNDS": ROUNDS, "MU": MU, "FRACOES": FRACOES, "DROPS": DROPS,
            "DROP_ROUNDS": DROP_ROUNDS, "N_TAU": N_TAU, "TAU_MIN": TAU_MIN, "TAU_MAX": TAU_MAX, "SEMENTE_MC": SEMENTE_MC,
            "K_GABARITO": K_GABARITO, "TAU_PSI": TAU_PSI, "VALIDACAO": VALIDACAO, "VALIDACAO_TAUS": VALIDACAO_TAUS,
            "VALIDACAO_ESCOLHIDO": VALIDACAO_ESCOLHIDO}


def json_com_estes_parametros():
    try:
        return json.loads(RESULTADOS_JSON.read_text())["parametros"] == json.loads(json.dumps(parametros_atuais()))
    except (OSError, KeyError, ValueError):
        return False


if DADOS_JSON:
    DADOS = json.loads(RESULTADOS_JSON.read_text())
    assert DADOS["parametros"] == json.loads(json.dumps(parametros_atuais())), f"a célula 1 difere do JSON, gerado com {DADOS['parametros']}"
    print(f"lendo {RESULTADOS_JSON.name} (gerado em {DADOS['gerado_em']}): nada será executado")
print("GPU:", torch.cuda.is_available(), "| processos do Monte Carlo:", PROCESSOS)

## 1. Execuções do script

`executar(α, seed, frac, drop, tau, rodadas)` roda o script uma vez com `--train 0 --k 1 --dirichlet_alpha α` e devolve a pasta de logs. Uma execução já concluída é reaproveitada se o `cluster.pkl` tiver o nº de rodadas pedido **e** a linha de argumentos do log tiver os mesmos parâmetros (o nome da pasta não traz o `τ`). A saída padrão vai para `logs/pipeline-varredura-*.txt`: ela tem os cossenos de cada rodada (linhas `matrix`), que o log principal não tem.

**Cuidado:** o nome da pasta de logs tem a hora só até o minuto. Duas execuções com os mesmos `n`, `m`, seed, `α` e `d` (e `τ` diferentes) iniciadas no mesmo minuto gravariam na mesma pasta, então `executar` espera o minuto mudar quando já existe uma pasta desse minuto.

In [ ]:
def dados_prontos(alpha, seed):
    d = PROJETO / f"datasets/rotated_mnist_{N}_{seed}_k1_a{alpha}"
    return len(list((d / "train").glob("data*.pkl"))) == N and (d / "test" / "data0.pkl").exists()


def pastas_de_log(alpha, m, seed, drop):
    tag = f"-drop{drop}-r{DROP_ROUNDS}" if drop > 0 else ""
    padrao = f"runs-rotated-mnist-n{N}-p{m}-seed{seed}-lambda{MU}-k1-a{alpha}{tag}-time-*"
    return sorted((PROJETO / "logs").glob(padrao), key=lambda p: p.stat().st_mtime)


def args_do_log(pasta):
    for linha in (pasta / "rotated_mnist.log").read_text().splitlines():
        if "Namespace(" in linha:
            return dict(re.findall(r"(\w+)=([^,()]+)", linha.split("Namespace(", 1)[1]))
    return {}


def concluida(pasta, alpha, m, seed, drop, tau, rodadas):
    try:
        if len(torch.load(pasta / "cluster.pkl", weights_only=False)) != rodadas:
            return False
        a = args_do_log(pasta)
        esperado = {"n": N, "com_round": rodadas, "num_per_round": m, "mu": MU, "tau": tau, "train": 0, "seed": seed, "k": 1,
                    "dirichlet_alpha": alpha, "drop": drop}
        if any(float(a[c]) != v for c, v in esperado.items()):
            return False
        return drop == 0 or int(a["drop_rounds"]) == DROP_ROUNDS
    except Exception:
        return False


def saida_de(alpha, m, seed, drop, tau, rodadas):
    return PROJETO / "logs" / f"pipeline-varredura-a{alpha}-m{m}-seed{seed}-drop{drop}-tau{tau}-r{rodadas}.txt"


def pronta(alpha, seed, frac, drop, tau, rodadas):
    m = int(N * frac)
    return next((p for p in reversed(pastas_de_log(alpha, m, seed, drop)) if concluida(p, alpha, m, seed, drop, tau, rodadas)), None)


def executar(alpha, seed, frac, drop, tau, rodadas):
    m = int(N * frac)
    if not REEXECUTAR and (pasta := pronta(alpha, seed, frac, drop, tau, rodadas)) is not None and saida_de(alpha, m, seed, drop, tau, rodadas).exists():
        return pasta, None
    # mesma combinação no mesmo minuto cairia na mesma pasta: espera o minuto mudar
    while any(p.name.endswith(time.strftime("-time-%m-%d-%H:%M")) for p in pastas_de_log(alpha, m, seed, drop)):
        time.sleep(5)
    cmd = [sys.executable, SCRIPT, "--n", str(N), "--com_round", str(rodadas), "--num_per_round", str(m), "--mu", str(MU),
           "--tau", str(tau), "--train", "0", "--seed", str(seed), "--dataset", "mnist", "--k", "1", "--dirichlet_alpha", str(alpha),
           "--process", "0" if dados_prontos(alpha, seed) else "1"]
    if drop > 0:
        cmd += ["--drop", str(drop), "--drop_rounds", str(DROP_ROUNDS)]
    (PROJETO / "logs").mkdir(exist_ok=True)
    saida = saida_de(alpha, m, seed, drop, tau, rodadas)
    inicio = time.time()
    with open(saida, "w") as f:
        r = subprocess.run(cmd, cwd=PROJETO, stdout=f, stderr=subprocess.STDOUT)
    if r.returncode != 0:
        print(saida.read_text()[-3000:])
        raise RuntimeError(f"falhou: {' '.join(cmd)}")
    pasta = next(p for p in reversed(pastas_de_log(alpha, m, seed, drop)) if p.stat().st_mtime >= inicio - 1)
    assert concluida(pasta, alpha, m, seed, drop, tau, rodadas), f"{pasta.name}: execução incompleta"
    return pasta, time.time() - inicio


def cossenos_por_rodada(saida):
    # linhas "matrix - INFO - a,b:cos" da saída padrão, agrupadas pela rodada (a linha "Starting round r" abre cada rodada)
    rodadas = []
    for linha in Path(saida).read_text().splitlines():
        if "Starting round" in linha:
            rodadas.append([])
        elif (achou := re.search(r"matrix - INFO - (\d+),(\d+):(-?[\d.]+)$", linha)):
            rodadas[-1].append((int(achou.group(1)), int(achou.group(2)), float(achou.group(3))))
    return rodadas

## 2. Cosseno entre clientes (`Ψ`)

Uma execução de 1 rodada com `m = 100%` e `τ = TAU_PSI` por `(α, seed)` observa os 240 clientes e grava `psi.pkl`. Dele sai `G = ΨΨᵀ` (float64). Conferências: a diagonal de `G` é 1 a menos do arredondamento (o `Ψ` é normalizado em float32 na GPU, e a norma sai 1 + ~1e-5) e os 28.680 cossenos da rodada 0 da saída padrão (6 casas) batem com o cosseno `G_ab / √(G_aa·G_bb)`, que é o que o script e o simulador usam, até 1e-4: o script calcula o cosseno em float32 na CPU (`cosine_sim`), com erro de até ~3e-5, e o simulador em float64. Por isso uma fusão só poderia divergir se um cosseno ficasse a menos de ~3e-5 de `τ`, o que a validação da seção 6 confere pelas partições. O simulador trabalha com `G` sem normalizar, porque o centro de um cluster é a média dos `Ψ` como estão.

A tabela é a **etapa A** da seção 6 do plano: a partição depois da rodada 0 com `m = 100%`, que é o grafo "cosseno > `τ`" entre clientes, para alguns `τ`.

In [ ]:
def gram_de(pasta):
    psi = torch.load(pasta / "psi.pkl", weights_only=False, map_location="cpu")
    assert sorted(psi) == list(range(N)), f"{pasta.name}: psi.pkl sem os {N} clientes"
    P = torch.stack([psi[c] for c in range(N)]).double()
    return (P @ P.T).numpy()


def cosseno(G):
    d = np.sqrt(np.diag(G))
    return G / np.outer(d, d)


def componentes(G, tau):
    # partição da rodada 0 com m = 100%: componentes conexas de "cosseno > tau", com o menor id de cada componente
    dono = np.arange(N)
    for a, b in zip(*np.nonzero(np.triu(cosseno(G) > tau, 1))):
        ra, rb = dono[a], dono[b]
        if ra != rb:
            dono[dono == max(ra, rb)] = min(ra, rb)
    return dono


def resumo_particao(dono):
    tam = np.bincount(np.unique(dono, return_inverse=True)[1])
    return {"clusters": int(len(tam)), "maior": int(tam.max()), "singletons": int((tam == 1).sum())}


combos_psi = [(a, s) for a in ALPHAS for s in SEEDS]
GRAM, PASTAS_PSI = {}, {}
if DADOS_JSON:
    for e in DADOS["clientes"]:
        G = np.zeros((N, N))
        G[np.triu_indices(N, 1)] = e["gram_triu"]
        G = G + G.T
        G[np.diag_indices(N)] = e["gram_diag"]
        GRAM[(e["alpha"], e["seed"])] = G
        PASTAS_PSI[(e["alpha"], e["seed"])] = e["pasta_psi"]
else:
    pendentes = [c for c in combos_psi if REEXECUTAR or pronta(c[0], c[1], 1.0, 0.0, TAU_PSI, 1) is None]
    if pendentes and not REEXECUTAR and json_com_estes_parametros():
        raise RuntimeError(f"{len(pendentes)} execuções sem logs, mas {RESULTADOS_JSON.name} já tem os resultados destes parâmetros: "
                           "use DADOS_JSON = True para plotar sem rodar nada, ou REEXECUTAR = True para rodar de novo")
    livre = shutil.disk_usage(PROJETO).free / 1e9
    print(f"{len(pendentes)} execuções de Ψ pendentes; espaço estimado {1.6 * len(pendentes) + 0.25 * len(pendentes):.0f} GB, livre {livre:.0f} GB")
    assert livre > 1.1 * (1.85 * len(pendentes) + 1.6 * (len(VALIDACAO) * len(VALIDACAO_TAUS) + 1))
    for i, (alpha, seed) in enumerate(combos_psi, start=1):
        pasta, dur = executar(alpha, seed, 1.0, 0.0, TAU_PSI, 1)
        G = gram_de(pasta)
        assert np.abs(np.diag(G) - 1).max() < 1e-4, "Ψ não normalizado"
        C = cosseno(G)
        cos0 = cossenos_por_rodada(saida_de(alpha, N, seed, 0.0, TAU_PSI, 1))[0]
        assert len(cos0) == N * (N - 1) // 2
        dif = max(abs(C[a, b] - c) for a, b, c in cos0)
        assert dif < 1e-4, f"cossenos da saída padrão diferem de G em {dif}"
        GRAM[(alpha, seed)], PASTAS_PSI[(alpha, seed)] = G, pasta.name
        estado = "reaproveitada" if dur is None else f"{dur:.0f} s"
        print(f"[{i}/{len(combos_psi)}] α={alpha} seed={seed}: {estado}, |saída − cosseno| máx {dif:.1e}, "
              f"|diag − 1| máx {np.abs(np.diag(G) - 1).max():.1e} -> {pasta.name}")

TAUS_A = [0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9]
etapa_a = pd.DataFrame([{"α": a, "seed": s, "τ": t, **resumo_particao(componentes(GRAM[(a, s)], t))}
                        for a, s in combos_psi for t in TAUS_A])
display(etapa_a.groupby(["α", "τ"])[["clusters", "maior", "singletons"]].agg(["mean", "min", "max"]).round(1))
iu = np.triu_indices(N, 1)
display(pd.DataFrame([{"α": a, "seed": s, "cos mediano": np.median(cosseno(GRAM[(a, s)])[iu]), "cos p90": np.quantile(cosseno(GRAM[(a, s)])[iu], 0.9),
                       "cos máx": cosseno(GRAM[(a, s)])[iu].max()} for a, s in combos_psi]).groupby("α").mean().drop(columns="seed").round(3))

## 3. Gabarito: agrupamento das distribuições de rótulos

Para cada `(α, seed)`, a proporção de cada classe em cada cliente (o histograma de rótulos normalizado, lido dos dados em `datasets/`) é agrupada por **Ward** (distância euclidiana), e o nº de grupos `k` é o de **maior silhueta** em `K_GABARITO`. O gabarito diz "clientes com composição de classes parecida", que é o que o `Ψ` deveria juntar; ele **não é uma verdade** e o ARI mede concordância com ele, não acerto.

In [ ]:
def histograma(alpha, seed):
    d = PROJETO / f"datasets/rotated_mnist_{N}_{seed}_k1_a{alpha}/train"
    return np.array([np.bincount(np.asarray(torch.load(d / f"data{c}.pkl", weights_only=False).y), minlength=10) for c in range(N)])


def gabarito_de(hist):
    props = hist / hist.sum(axis=1, keepdims=True)
    Z = linkage(props, "ward")
    sil = {k: float(silhouette_score(props, fcluster(Z, k, "maxclust"))) for k in K_GABARITO}
    k = max(sil, key=sil.get)
    return (fcluster(Z, k, "maxclust") - 1).tolist(), k, sil


HIST, GABARITO, K_GAB, SILHUETA = {}, {}, {}, {}
for alpha, seed in combos_psi:
    if DADOS_JSON:
        e = next(e for e in DADOS["clientes"] if (e["alpha"], e["seed"]) == (alpha, seed))
        HIST[(alpha, seed)], GABARITO[(alpha, seed)] = np.array(e["histograma"]), e["gabarito"]
        K_GAB[(alpha, seed)], SILHUETA[(alpha, seed)] = e["k_gabarito"], {int(k): v for k, v in e["silhueta"].items()}
    else:
        HIST[(alpha, seed)] = histograma(alpha, seed)
        GABARITO[(alpha, seed)], K_GAB[(alpha, seed)], SILHUETA[(alpha, seed)] = gabarito_de(HIST[(alpha, seed)])

tab_gab = pd.DataFrame([{"α": a, "seed": s, "k": K_GAB[(a, s)], "silhueta": SILHUETA[(a, s)][K_GAB[(a, s)]],
                         "maior grupo": int(np.bincount(GABARITO[(a, s)]).max()), "menor grupo": int(np.bincount(GABARITO[(a, s)]).min())}
                        for a, s in combos_psi])
display(tab_gab.groupby("α").agg(k_medio=("k", "mean"), k_min=("k", "min"), k_max=("k", "max"), silhueta=("silhueta", "mean"),
                                 maior_grupo=("maior grupo", "mean"), menor_grupo=("menor grupo", "mean")).round(3))

## 4. Simulador da clusterização e validação contra o script

`sorteios(seed, m, d)` repete o sorteio do script: `random.seed(seed)`, e, com dropout, `Availability(N, d, DROP_ROUNDS, seed).step()` e o sorteio de `min(m, disponíveis)` entre os disponíveis. O sorteio não depende de `τ`, então é calculado uma vez por `(seed, m, d)`.

`simular(G, sorteados, tau)` repete a fusão do script, rodada a rodada, e devolve o cluster de cada cliente em cada rodada (o id é o menor cliente do cluster, como no `cluster.pkl`).

A validação roda o script real (50 rodadas) nas combinações de `VALIDACAO` × `VALIDACAO_TAUS` e mais uma com o `τ` escolhido (seção 5), e confere, **em todas as rodadas**: (1) os sorteados e os indisponíveis; (2) a partição; (3) os cossenos entre os clusters sorteados (da saída padrão, 6 casas) contra os do simulador.

In [ ]:
def sorteios(seed, m, drop):
    rng = random.Random(seed)   # o script faz random.seed(seed) em setup_seed e só usa o random global no sorteio
    disp = Availability(N, drop, DROP_ROUNDS, seed)
    clientes = list(range(N))
    sorteados, indisponiveis = [], []
    for _ in range(ROUNDS):
        if drop > 0:
            up = disp.step()
            livres = [c for c in clientes if up[c]]
            indisponiveis.append([c for c in clientes if not up[c]])
            sorteados.append(sorted(rng.sample(livres, min(m, len(livres)))))
        else:
            indisponiveis.append([])
            sorteados.append(sorted(rng.sample(clientes, m)))
    return sorteados, indisponiveis


def simular(G, sorteados, tau, cossenos=False):
    # R[a] = soma das linhas de G dos membros do cluster a (vale para clusters vivos); S_AB = soma de R[A] nas colunas de B
    dono = np.arange(N)
    membros = {i: [i] for i in range(N)}
    R = G.copy()
    rodadas, cos_rodadas = [], []
    for cid in sorteados:
        amostrados = sorted({int(dono[c]) for c in cid})
        pares = []
        if len(amostrados) > 1 and len(membros) > 1:
            Q = np.zeros((len(amostrados), N))
            for i, a in enumerate(amostrados):
                Q[i, membros[a]] = 1.0
            S = R[amostrados] @ Q.T
            d = np.sqrt(np.diag(S))
            C = S / np.outer(d, d)
            ii, jj = np.triu_indices(len(amostrados), 1)
            if cossenos:
                pares = [(amostrados[i], amostrados[j], C[i, j]) for i, j in zip(ii, jj)]
            sel = C[ii, jj] > tau
            if sel.any():
                # componentes conexas entre os clusters sorteados (union-find), cada uma fica com o menor id
                pai = {a: a for a in amostrados}
                def raiz(a):
                    while pai[a] != a:
                        a = pai[a]
                    return a
                for i, j in zip(ii[sel], jj[sel]):
                    ra, rb = raiz(amostrados[i]), raiz(amostrados[j])
                    if ra != rb:
                        pai[max(ra, rb)] = min(ra, rb)
                for a in amostrados:
                    r = raiz(a)
                    if r != a:
                        membros[r] += membros.pop(a)
                        R[r] += R[a]
                        dono[membros[r]] = r
        rodadas.append(dono.copy())
        cos_rodadas.append(pares)
    return (rodadas, cos_rodadas) if cossenos else rodadas


combos_sorteio = [(s, int(N * f), d) for s in SEEDS for f in FRACOES for d in DROPS]
if DADOS_JSON:
    SORTEIOS = {(e["seed"], e["m"], e["drop"]): (e["sorteados_por_rodada"], e["indisponiveis_por_rodada"]) for e in DADOS["sorteios"]}
else:
    SORTEIOS = {c: sorteios(*c) for c in combos_sorteio}


def validar(alpha, seed, frac, drop, tau):
    # roda o script real e compara com o simulador, rodada a rodada
    m = int(N * frac)
    pasta, dur = executar(alpha, seed, frac, drop, tau, ROUNDS)
    log = torch.load(pasta / "cluster.pkl", weights_only=False)
    real = []
    for particao in log:
        r = np.empty(N, dtype=int)
        for cid, mem in particao.items():
            r[mem] = cid
        real.append(r.tolist())
    sort_real = [[int(x) for x in a.group(1).split(",") if x.strip()]
                 for a in re.finditer(r"client id this round \[(.*?)\]", (pasta / "rotated_mnist.log").read_text())]
    arq = pasta / "availability.pkl"
    indisp_real = torch.load(arq, weights_only=False) if arq.exists() else [[] for _ in log]
    sort_sim, indisp_sim = SORTEIOS[(seed, m, drop)]
    sim, cos_sim = simular(GRAM[(alpha, seed)], sort_sim, tau, cossenos=True)
    cos_real = cossenos_por_rodada(saida_de(alpha, m, seed, drop, tau, ROUNDS))
    # o script só grava cossenos nas rodadas com 2 clusters sorteados ou mais, como o simulador
    dif_cos, pares_iguais = 0.0, True
    for cr, cs in zip(cos_real, cos_sim):
        dr, ds = {(a, b): c for a, b, c in cr}, {(a, b): c for a, b, c in cs}
        pares_iguais &= dr.keys() == ds.keys()
        if dr.keys() == ds.keys() and dr:
            dif_cos = max(dif_cos, max(abs(dr[k] - ds[k]) for k in dr))
    return {"alpha": alpha, "seed": seed, "m": m, "drop": drop, "tau": tau, "pasta": pasta.name,
            "sorteados_iguais": sort_real == sort_sim, "indisponiveis_iguais": [list(map(int, u)) for u in indisp_real] == indisp_sim,
            "particoes_iguais": real == [r.tolist() for r in sim], "pares_de_cosseno_iguais": bool(pares_iguais),
            "dif_max_cosseno": float(dif_cos), "rodadas": len(log),
            "clusters_final": resumo_particao(np.array(real[-1]))["clusters"],
            "rotulos_por_rodada": real, "cos_max_por_rodada": [max((c for *_, c in cr), default=None) for cr in cos_real],
            "n_pares_cos_por_rodada": [len(cr) for cr in cos_real], "duracao_s": dur}

## 5. Monte Carlo sobre τ

Para cada `τ` sorteado e cada uma das 120 combinações `(α, seed, m, d)`, simula as 50 rodadas e mede a partição final contra o gabarito:

| Métrica | Significado |
|---|---|
| `ari` | ARI entre a partição final e o gabarito (seção 3) |
| `clusters`, `maior`, `singletons` | nº de clusters, tamanho do maior e nº de clusters de 1 cliente |
| `tvd` | média, entre os clientes, da distância de variação total entre a distribuição de rótulos do cliente e a do seu cluster (soma dos histogramas dos membros); 0 = cada cliente igual ao seu cluster |
| `cobertura` | fração de clientes sorteados ao menos uma vez |

O `τ` escolhido é o de **maior ARI médio** nas 120 combinações (D-046). Como o resultado é uma função em degraus de `τ`, ele é arredondado para 3 casas só se o valor arredondado der o mesmo ARI médio.

In [ ]:
TAUS = np.sort(np.random.default_rng(SEMENTE_MC).uniform(TAU_MIN, TAU_MAX, N_TAU))
unidades = [(a, s, f, d) for a in ALPHAS for s in SEEDS for f in FRACOES for d in DROPS]


def metricas_finais(dono, alpha, seed, sorteados):
    hist = HIST[(alpha, seed)]
    _, inv = np.unique(dono, return_inverse=True)
    hc = np.zeros((inv.max() + 1, 10))
    np.add.at(hc, inv, hist)
    p_cli = hist / hist.sum(1, keepdims=True)
    p_clu = hc[inv] / hc[inv].sum(1, keepdims=True)
    return {"ari": float(adjusted_rand_score(GABARITO[(alpha, seed)], dono)), **resumo_particao(dono),
            "tvd": float(0.5 * np.abs(p_cli - p_clu).sum(1).mean()), "cobertura": len(set().union(*map(set, sorteados))) / N}


def rodar_unidade(unidade, taus=None):
    alpha, seed, frac, drop = unidade
    sorteados = SORTEIOS[(seed, int(N * frac), drop)][0]
    linhas = []
    with threadpool_limits(1):
        for tau in (TAUS if taus is None else taus):
            dono = simular(GRAM[(alpha, seed)], sorteados, tau)[-1]
            linhas.append({"tau": float(tau), "alpha": alpha, "seed": seed, "m": frac, "drop": drop, **metricas_finais(dono, alpha, seed, sorteados)})
    return linhas


def rodar_mc(taus=None):
    with ProcessPoolExecutor(PROCESSOS, mp_context=multiprocessing.get_context("fork")) as ex:   # fork: os processos herdam GRAM, HIST, GABARITO e SORTEIOS
        return pd.DataFrame([l for ls in ex.map(rodar_unidade, unidades, [taus] * len(unidades)) for l in ls])


if DADOS_JSON:
    mc = pd.DataFrame(DADOS["monte_carlo"])
    escolha = DADOS["escolha"]
else:
    inicio = time.time()
    mc = rodar_mc()
    print(f"Monte Carlo: {len(TAUS)} τ × {len(unidades)} combinações = {len(mc)} simulações em {time.time() - inicio:.0f} s")
    media = mc.groupby("tau")["ari"].mean()
    tau_max = float(media.idxmax())
    # arredonda se o τ arredondado der o mesmo ARI médio (3 casas; senão 4, 5, ...)
    for casas in range(3, 8):
        tau_r = round(tau_max, casas)
        ari_r = rodar_mc([tau_r])["ari"].mean()
        if abs(ari_r - media.max()) < 1e-12:
            break
    vizinhos = media[media >= media.max() - 0.01]
    escolha = {"tau_sorteado": tau_max, "tau": tau_r, "ari_medio": float(media.max()),
               "taus_ate_0.01_do_max": [float(vizinhos.index.min()), float(vizinhos.index.max())], "n_taus_ate_0.01_do_max": int(len(vizinhos))}
TAU_ESCOLHIDO = escolha["tau"]
print(f"τ escolhido = {TAU_ESCOLHIDO} (ARI médio {escolha['ari_medio']:.4f}); τ sorteados a até 0,01 do máximo: "
      f"{escolha['n_taus_ate_0.01_do_max']}, de {escolha['taus_ate_0.01_do_max'][0]:.4f} a {escolha['taus_ate_0.01_do_max'][1]:.4f}")

por_tau = mc.groupby("tau")[["ari", "clusters", "maior", "singletons", "tvd"]].mean()
# o τ de maior ARI médio dentro de cada α, só para comparação (a grade usa um τ só)
por_alpha = mc.groupby(["alpha", "tau"])["ari"].mean().reset_index()
display(pd.DataFrame([{"α": a, "τ de maior ARI no α": float(g.loc[g["ari"].idxmax(), "tau"]), "ARI máx": g["ari"].max(),
                       f"ARI em τ = {TAU_ESCOLHIDO}": g.loc[(g["tau"] - escolha["tau_sorteado"]).abs().idxmin(), "ari"]}
                      for a, g in por_alpha.groupby("alpha")]).round(4))

### Tabela por τ

Médias nas 120 combinações para os `τ` sorteados mais próximos de alguns valores de referência e do `τ` escolhido, e a partição em `τ` escolhido por `α`, `m` e cenário de dropout (média e faixa entre as seeds).

In [ ]:
refs = sorted(set([0.5, 0.6, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, escolha["tau_sorteado"]]))
idx = [por_tau.index[np.abs(por_tau.index - r).argmin()] for r in refs]
display(por_tau.loc[idx].rename_axis("τ sorteado").round(3))

no_tau = mc[np.isclose(mc["tau"], escolha["tau_sorteado"])]
tab_escolha = no_tau.groupby(["alpha", "m", "drop"]).agg(
    ari=("ari", "mean"), ari_min=("ari", "min"), ari_max=("ari", "max"), clusters=("clusters", "mean"), clusters_min=("clusters", "min"),
    clusters_max=("clusters", "max"), maior=("maior", "mean"), singletons=("singletons", "mean"), tvd=("tvd", "mean"), cobertura=("cobertura", "mean")).reset_index()
tab_escolha.insert(3, "cenário", tab_escolha.pop("drop").map(nome))
tab_escolha["m"] = tab_escolha["m"].map(lambda f: f"{f:.0%}")
display(tab_escolha.round(3))

### Gráficos

Média nas seeds, `m` e cenários de dropout de cada `α`, em função de `τ` (um ponto por `τ` sorteado); a linha vertical marca o `τ` escolhido. O nº de clusters está em escala log (vai de 1 a 240).

In [ ]:
CORES_A = ["#2a78d6", "#eb6834"]
paineis = [("ari", "ARI contra o gabarito"), ("clusters", "Nº de clusters (log)"), ("maior", "Maior cluster"), ("tvd", "TVD cliente–cluster")]
fig = make_subplots(rows=2, cols=2, subplot_titles=[t for _, t in paineis], horizontal_spacing=0.09, vertical_spacing=0.14)
for i, (col, titulo) in enumerate(paineis):
    r, c = i // 2 + 1, i % 2 + 1
    for j, a in enumerate(ALPHAS):
        g = mc[mc["alpha"] == a].groupby("tau")[col].mean()
        fig.add_trace(go.Scatter(x=g.index, y=g.values, mode="lines+markers", name=f"α = {a}", legendgroup=str(a), showlegend=(i == 0),
                                 line=dict(color=CORES_A[j], width=2), marker=dict(size=4),
                                 hovertemplate="τ = %{x:.4f}: %{y:.3g}<extra>α = " + str(a) + "</extra>"), row=r, col=c)
    fig.add_vline(x=TAU_ESCOLHIDO, line=dict(color="gray", dash="dot", width=1), row=r, col=c)
    fig.update_xaxes(title_text="τ", row=r, col=c)
fig.update_yaxes(type="log", row=1, col=2)
fig.update_layout(title=f"Monte Carlo sobre τ: {N_TAU} τ × {len(unidades)} combinações; τ escolhido = {TAU_ESCOLHIDO}", height=700,
                  hovermode="x unified", template="simple_white", legend=dict(orientation="h", y=-0.1), margin=dict(t=90, b=60, l=50, r=20))
fig.update_xaxes(showgrid=True, gridcolor="rgba(128,128,128,0.2)")
fig.update_yaxes(showgrid=True, gridcolor="rgba(128,128,128,0.2)")
fig.show()

# ARI por m e cenário de dropout, α a α
fig = make_subplots(rows=1, cols=len(ALPHAS), subplot_titles=[f"α = {a}" for a in ALPHAS], horizontal_spacing=0.08)
CORES_D = ["#2a78d6", "#eb6834", "#1baf7a"]
TRACOS_M = {1.0: "solid", 0.5: "dash"}
for j, a in enumerate(ALPHAS, start=1):
    for f in FRACOES:
        for k, d in enumerate(DROPS):
            g = mc[(mc["alpha"] == a) & (mc["m"] == f) & (mc["drop"] == d)].groupby("tau")["ari"].mean()
            leg = f"m = {f:.0%}, {nome(d)}"
            fig.add_trace(go.Scatter(x=g.index, y=g.values, mode="lines", name=leg, legendgroup=leg, showlegend=(j == 1),
                                     line=dict(color=CORES_D[k], dash=TRACOS_M[f], width=2),
                                     hovertemplate="τ = %{x:.4f}: %{y:.3f}<extra>" + leg + "</extra>"), row=1, col=j)
    fig.add_vline(x=TAU_ESCOLHIDO, line=dict(color="gray", dash="dot", width=1), row=1, col=j)
    fig.update_xaxes(title_text="τ", row=1, col=j)
fig.update_yaxes(title_text="ARI médio nas seeds", row=1, col=1)
fig.update_layout(title="ARI contra o gabarito por m e cenário de dropout", height=450, hovermode="x unified", template="simple_white",
                  legend=dict(orientation="h", y=-0.2), margin=dict(t=80, b=80, l=60, r=20))
fig.show()

## 6. Validação do simulador

Roda o script real nas combinações de validação (a última com o `τ` escolhido) e compara rodada a rodada. Todas as colunas `*_iguais` têm de ser `True` e a diferença máxima de cosseno tem de ficar abaixo de `1e-4`; senão, o Monte Carlo não representa o script e o `τ` escolhido não vale.

In [ ]:
combos_val = [(a, s, f, d, t) for (a, s, f, d) in VALIDACAO for t in VALIDACAO_TAUS] + [(*VALIDACAO_ESCOLHIDO, TAU_ESCOLHIDO)]
if DADOS_JSON:
    validacao = DADOS["validacao"]
else:
    validacao = []
    for i, c in enumerate(combos_val, start=1):
        v = validar(*c)
        validacao.append(v)
        estado = "reaproveitada" if v["duracao_s"] is None else f"{v['duracao_s']:.0f} s"
        print(f"[{i}/{len(combos_val)}] α={c[0]} seed={c[1]} m={c[2]:.0%} {nome(c[3])} τ={c[4]}: {estado}")
tab_val = pd.DataFrame([{k: v for k, v in e.items() if k not in ("rotulos_por_rodada", "cos_max_por_rodada", "n_pares_cos_por_rodada")} for e in validacao])
display(tab_val)
assert tab_val[["sorteados_iguais", "indisponiveis_iguais", "particoes_iguais", "pares_de_cosseno_iguais"]].all().all(), "o simulador diverge do script"
assert (tab_val["dif_max_cosseno"] < 1e-4).all()
print("simulador igual ao script em todas as rodadas das", len(tab_val), "execuções de validação")

## 7. Resultados em JSON

Grava `logs/resultados-varredura-tau-dirichlet.json` (estrito, via arquivo temporário; nesse modo `DADOS_JSON` não grava nada):

- `parametros` e `escolha` (o `τ` escolhido, o sorteado de onde veio, o ARI médio e a faixa de `τ` a até 0,01 do máximo).
- `clientes`: por `(α, seed)`, o triângulo superior e a diagonal de `G` (cosseno entre clientes, float64), o histograma de rótulos (240 × 10), o gabarito, o seu `k` e a silhueta de cada `k`, e a pasta de logs dos `Ψ` (só histórico).
- `sorteios`: por `(seed, m, d)`, os sorteados e os indisponíveis de cada rodada.
- `monte_carlo`: uma linha por `τ` e combinação, com as métricas da seção 5. Com `clientes` e `sorteios`, o simulador refaz qualquer linha.
- `validacao`: as comparações da seção 6, com a partição real de cada rodada (`rotulos_por_rodada`), o maior cosseno e o nº de pares de cada rodada da saída padrão.
- `etapa_a`: a tabela da seção 2.

In [ ]:
def registros(df):
    # precisão completa (o df.to_json do pandas grava só 10 algarismos); numpy -> tipos do Python
    return [{k: (v.item() if hasattr(v, "item") else v) for k, v in r.items()} for r in df.to_dict("records")]


# consistência: refaz com o simulador algumas linhas do Monte Carlo a partir dos dados que vão (ou foram) para o JSON
amostra = mc.sample(5, random_state=0)
for _, l in amostra.iterrows():
    sort = SORTEIOS[(int(l["seed"]), int(N * l["m"]), float(l["drop"]))][0]
    dono = simular(GRAM[(float(l["alpha"]), int(l["seed"]))], sort, float(l["tau"]))[-1]
    assert abs(metricas_finais(dono, float(l["alpha"]), int(l["seed"]), sort)["ari"] - l["ari"]) < 1e-12
print("5 linhas do Monte Carlo refeitas pelo simulador: iguais")

if DADOS_JSON:
    print(f"dados lidos de {RESULTADOS_JSON.name}: nada a gravar")
else:
    resultado = {
        "gerado_em": datetime.now().isoformat(timespec="seconds"),
        "parametros": parametros_atuais(),
        "escolha": escolha,
        "clientes": [{"alpha": a, "seed": s, "pasta_psi": PASTAS_PSI[(a, s)], "gram_triu": GRAM[(a, s)][np.triu_indices(N, 1)].tolist(), "gram_diag": np.diag(GRAM[(a, s)]).tolist(),
                      "histograma": HIST[(a, s)].tolist(), "gabarito": [int(g) for g in GABARITO[(a, s)]], "k_gabarito": int(K_GAB[(a, s)]),
                      "silhueta": {str(k): v for k, v in SILHUETA[(a, s)].items()}} for a, s in combos_psi],
        "sorteios": [{"seed": s, "m": m, "drop": d, "sorteados_por_rodada": SORTEIOS[(s, m, d)][0], "indisponiveis_por_rodada": SORTEIOS[(s, m, d)][1]}
                     for s, m, d in combos_sorteio],
        "monte_carlo": registros(mc),
        "validacao": validacao,
        "etapa_a": registros(etapa_a),
    }
    tmp = RESULTADOS_JSON.with_suffix(".json.tmp")
    tmp.write_text(json.dumps(resultado, ensure_ascii=False, allow_nan=False))
    tmp.replace(RESULTADOS_JSON)
    print(f"{RESULTADOS_JSON} ({RESULTADOS_JSON.stat().st_size / 1e6:.1f} MB)")

## 8. Leitura

- **O `τ` escolhido vale para o gabarito escolhido.** O ARI mede a concordância com um agrupamento dos histogramas (Ward + silhueta), que é uma construção. Outro gabarito (outra ligação, outro `k`) poderia apontar outro `τ`.
- **Degraus.** A partição só muda quando `τ` cruza um cosseno entre clusters, então o ARI médio é uma função em degraus de `τ`; a faixa de `τ` a até 0,01 do máximo mostra quão largo é o ótimo.
- **Um `τ` para os dois `α`.** A grade usa um só `τ`; a tabela da seção 5 mostra o `τ` que cada `α` escolheria sozinho.
- **Sem treino.** Com `--train 1` a partição é a mesma (calibração do plano, seção 3), então o `τ` vale para a grade com treino.